# 🔭 PANOSETI Run Preview

This notebook provides a convenience interface to preview the latest data products and configurations from a PANOSETI observing run.

In [ ]:
%matplotlib inline
import os
import glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pypff import PanosetiRun, PFFSequence, hkpff
import plotly.graph_objects as go
from tqdm.notebook import tqdm

# --- Configuration ---
DATA_DIR = "/data/panoseti"  # Base directory containing .pffd runs
# -----------------------

## 1. Discover Latest Run

Automatically finding the most recent `.pffd` directory in the configured data path.

In [ ]:
def get_latest_run(base_dir):
    runs = sorted(glob.glob(os.path.join(base_dir, "*.pffd")))
    if not runs:
        return None
    return Path(runs[-1])

latest_run_path = get_latest_run(DATA_DIR)

if latest_run_path:
    print(f"✅ Latest Run Found: {latest_run_path.name}")
    run = PanosetiRun(latest_run_path)
    run.show()
else:
    print(f"❌ No runs found in {DATA_DIR}")

## 2. Interactive Data Product Explorer

Browse through images and pulse-height plots with detailed header metadata.

In [ ]:
if 'run' in locals():
    products = run.list_products()
    
    product_dropdown = widgets.Dropdown(
        options=products,
        description='Product:',
        layout=widgets.Layout(width='400px')
    )

    frame_slider = widgets.IntSlider(
        min=0,
        max=100,
        step=1,
        description='Frame:',
        continuous_update=False,
        layout=widgets.Layout(width='600px')
    )

    out = widgets.Output()

    def format_header(header):
        """Helper to format PFF/Quabo header into a nice string."""
        if hasattr(header, 'quabo_num'):
            # Single Quabo Header (PH256)
            return (
                f"Quabo {header.quabo_num} | Pkt: {header.pkt_num} | "
                f"TAI: {header.pkt_tai} | ns: {header.pkt_nsec}\n"
                f"UTC: {header.tv_sec}.{header.tv_usec:06d}"
            )
        elif hasattr(header, 'quabo_0'):
            # Module Header (Image)
            return (
                f"Module Group | TAI (Q0): {header.quabo_0.pkt_tai} | ns: {header.quabo_0.pkt_nsec}\n"
                f"UTC: {header.quabo_0.tv_sec}.{header.quabo_0.tv_usec:06d}"
            )
        return str(header)

    def update_display(change=None):
        product_name = product_dropdown.value
        seq = run.get_product(product_name)
        frame_slider.max = len(seq) - 1
        
        frame_idx = frame_slider.value
        header, data = seq.get_frame(frame_idx)
        
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            
            if "ph" in product_name:
                # Pulse Height Plot
                ax.plot(data.flatten(), drawstyle='steps-mid', color='tab:blue')
                ax.set_ylabel("ADC")
                ax.set_xlabel("Pixel")
                ax.grid(True, alpha=0.3)
            else:
                # Image Plot
                im = ax.imshow(data, cmap='magma', origin='lower')
                fig.colorbar(im, ax=ax, label='ADC Value')
            
            title = f"{product_name} [Frame {frame_idx}]\n{format_header(header)}"
            ax.set_title(title, loc='left', family='monospace', fontsize=10)
            plt.tight_layout()
            plt.show()

    product_dropdown.observe(update_display, names='value')
    frame_slider.observe(update_display, names='value')
    update_display()

    display(widgets.VBox([product_dropdown, frame_slider, out]))
else:
    print("Run not loaded. Cannot display explorer.")

## 3. Configuration & Metadata Preview

Quickly inspect the Pydantic-validated configurations stored with the run.

In [ ]:
if 'run' in locals():
    config_select = widgets.Dropdown(
        options=sorted(run.configs.keys()),
        description='Config:',
        layout=widgets.Layout(width='400px')
    )
    
    config_out = widgets.Output()
    
    def show_config(change=None):
        with config_out:
            clear_output(wait=True)
            cfg_name = config_select.value
            cfg = run.configs[cfg_name]
            # If it's a Pydantic model, use model_dump; else print raw
            if hasattr(cfg, 'model_dump_json'):
                print(cfg.model_dump_json(indent=2))
            else:
                import json
                print(json.dumps(cfg, indent=2))
                
    config_select.observe(show_config, names='value')
    show_config()
    display(widgets.VBox([config_select, config_out]))